# Softmax 手撕实现

## 1. 定义
对向量 $z=(z_1,\dots,z_n)$，softmax 把它归一化为概率分布：
$$\text{softmax}(z_i)=\frac{e^{z_i}}{\sum_j e^{z_j}}$$
性质：输出非负且和为 1；输入平移常数 $c$ 结果不变（$\text{softmax}(z)=\text{softmax}(z-c)$），这是数值稳定的关键。

## 2. 数值溢出问题
$e^{z}$ 当 $z$ 较大时（如 $z=1000$）会 inf。fp16 上 $e^{12}$ 已接近上溢。朴素实现 `exp(z)/sum(exp(z))` 会得到 `inf/inf=nan`。

## 3. Safe Softmax（减最大值）
利用平移不变性，先减去 $m=\max(z)$：
$$\text{softmax}(z_i)=\frac{e^{z_i-m}}{\sum_j e^{z_j-m}}$$
此时最大指数项为 $e^0=1$，不会上溢；所有指数项 $\le 1$。

## 4. Online Softmax（FlashAttention 的基础）
流式处理：逐元素更新行最大值 $m$ 与分母 $d$，无需一次性拿到全部数据，便于分块（tiling）：
- 新来 $z_i$：$m_{new}=\max(m, z_i)$，$d\leftarrow d\cdot e^{m-m_{new}} + e^{z_i-m_{new}}$
- 最终 $\text{softmax}(z_i)=e^{z_i-m}/d$

## 5. 导数
$\frac{\partial s_i}{\partial z_j}=s_i(\delta_{ij}-s_j)$，雅可比可写成 $\text{diag}(s)-ss^T$。反向时 $\nabla z = s\odot(\nabla s - \text{sum}(s\odot\nabla s))$。

In [ ]:
import numpy as np
import torch

# 朴素 softmax：演示 fp16 上溢
z = np.array([0, 7, 6, 12, 10], dtype=np.float16)
ez = np.exp(z)
print('exp(z):', ez)            # 12 处已 inf
print('naive :', ez / np.sum(ez))  # inf/inf -> nan

In [ ]:
# Safe softmax：减最大值
def softmax_safe(z):
    m = np.max(z)
    e = np.exp(z - m)
    return e / np.sum(e)

print('safe  :', softmax_safe(z.astype(np.float32)))
print('torch :', torch.softmax(torch.tensor(z, dtype=torch.float32), dim=0).numpy())

In [ ]:
# Online softmax（单次遍历更新 m, d；再单次遍历归一化）
def softmax_online(z):
    L = len(z)
    m = float('-inf')
    d = 0.0
    for zi in z:
        m_new = max(m, zi)
        d = d * np.exp(m - m_new) + np.exp(zi - m_new)
        m = m_new
    return np.exp(z - m) / d

z32 = z.astype(np.float32)
print('online:', softmax_online(z32))
print('match  :', np.allclose(softmax_online(z32), softmax_safe(z32)))

In [ ]:
# 导数验证：雅可比 J = diag(s) - s s^T
z = torch.randn(4, requires_grad=True)
s = torch.softmax(z, dim=0)
loss = s.sum()          # 取 sum 作为标量目标，等价于 grad_out = ones
loss.backward()
# 解析解：dz_i = s_i * (1 - sum(s)) = s_i*(1-1) = 0
print('autograd dz:', z.grad)
print('analytic   :', s * (1 - s.sum()))

## 小结 / 易错点
- **必须减最大值**，否则 fp16/fp32 大输入都会 inf。
- **online 形式**是 FlashAttention 的核心：把 softmax 拆成"逐块更新 $m,d$"，避免物化 $N\times N$ 矩阵。
- 反向梯度公式 $\nabla z = s\odot(\nabla s - \sum(s\odot\nabla s))$ 在手写 attention 反向时常用。
- fp16 下 $e^{11.09}\approx 65504$ 已是上界，长序列 attention 的 $qk^T/\sqrt d$ 极易溢出，故需 safe softmax + 缩放。